# Perfilado de Datos — Cruz Azul F.C. Intelligence
**Materia:** Calidad y Preprocesamiento de Datos  
**Framework:** DAMA-DMBOK  
**Etapa:** I — Perfilado (antes y después de limpieza)  

Este notebook realiza el perfilado de las 4 fuentes de datos utilizadas en el proyecto:
1. **LigaMX** — Partidos 2016-2024 (formato JSON)
2. **Transfermarkt** — Jugadores (formato XLSX)
3. **FIFA 23** — Entrenadores masculinos (formato TXT)
4. **OpenPublicDomainFootballData** — Partidos históricos Liga MX (formato CSV)

---
### Dimensiones DAMA-DMBOK evaluadas:
| Dimensión | Qué medimos |
|---|---|
| Completitud | % de valores nulos por columna |
| Unicidad | % de registros duplicados |
| Validez | Tipos de dato, patrones, rangos |
| Consistencia | Formatos heterogéneos entre fuentes |

In [ ]:
# Instalar dependencias si no están disponibles
!pip install ydata-profiling openpyxl pyarrow rapidfuzz -q

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import seaborn as sns
import glob
import os
import warnings
warnings.filterwarnings('ignore')

# Estilo de gráficas
plt.rcParams['figure.figsize'] = (12, 5)
plt.rcParams['axes.titlesize'] = 13
sns.set_style('whitegrid')
COLORES = ['#003DA5', '#C8102E', '#FFFFFF', '#888888']  # Paleta Cruz Azul

# Rutas base
BASE       = os.path.abspath(os.path.join(os.getcwd(), '..'))
RAW        = os.path.join(BASE, 'Datos_originales')
MOD        = os.path.join(BASE, 'Datos_modificados')
REPORTS    = os.path.join(BASE, 'reports')
os.makedirs(REPORTS, exist_ok=True)

print('Rutas configuradas:')
print(f'  BASE    : {BASE}')
print(f'  MOD     : {MOD}')
print(f'  REPORTS : {REPORTS}')

---
## 1. Carga de datos desde múltiples formatos

Las 4 fuentes se encuentran en formatos distintos (**JSON, XLSX, TXT, CSV**), lo que refleja la heterogeneidad real de los sistemas de información deportiva.

In [ ]:
# ── Fuente 1: LigaMX → JSON ──────────────────────────────────────────────────
ruta_ligamx = os.path.join(MOD, 'LigaMX', '2016-2024_liga_mx.json')
df_ligamx = pd.read_json(ruta_ligamx)
print(f'LigaMX cargado desde JSON: {df_ligamx.shape[0]:,} filas x {df_ligamx.shape[1]} columnas')

# ── Fuente 2: Transfermarkt players → XLSX ───────────────────────────────────
ruta_tm = os.path.join(MOD, 'FootballDatafromTransfermarkt', 'players.xlsx')
df_tm = pd.read_excel(ruta_tm, engine='openpyxl')
print(f'Transfermarkt cargado desde XLSX: {df_tm.shape[0]:,} filas x {df_tm.shape[1]} columnas')

# ── Fuente 3: FIFA 23 male_coaches → TXT (tab-separated) ─────────────────────
ruta_fifa = os.path.join(MOD, 'FIFA23complete_player_dataset', 'male_coaches.txt')
df_fifa = pd.read_csv(ruta_fifa, sep='\t', low_memory=False)
print(f'FIFA23 cargado desde TXT: {df_fifa.shape[0]:,} filas x {df_fifa.shape[1]} columnas')

# ── Fuente 4: OpenPublicDomain (todos los mx.1.csv de cada temporada) ─────────
archivos_mx = sorted(glob.glob(os.path.join(RAW, 'OpenPublicDomainFootballData', '*', 'mx.1.csv')))
df_opfd = pd.concat([pd.read_csv(f) for f in archivos_mx], ignore_index=True)
print(f'OpenPublicDomain cargado desde CSV ({len(archivos_mx)} temporadas): {df_opfd.shape[0]:,} filas x {df_opfd.shape[1]} columnas')

---
## 2. Función auxiliar de perfilado

Centralizamos las métricas de calidad en una función reutilizable alineada con las dimensiones DAMA-DMBOK.

In [ ]:
def perfil_basico(df, nombre_fuente):
    """Imprime resumen de calidad de un DataFrame."""
    total_celdas = df.shape[0] * df.shape[1]
    nulos_total  = df.isnull().sum().sum()
    duplicados   = df.duplicated().sum()

    print(f'\n{'='*60}')
    print(f'  FUENTE: {nombre_fuente}')
    print(f'{'='*60}')
    print(f'  Filas              : {df.shape[0]:>10,}')
    print(f'  Columnas           : {df.shape[1]:>10}')
    print(f'  Celdas totales     : {total_celdas:>10,}')
    print(f'  Celdas nulas       : {nulos_total:>10,}  ({nulos_total/total_celdas*100:.1f}%)')
    print(f'  Filas duplicadas   : {duplicados:>10,}  ({duplicados/df.shape[0]*100:.1f}%)')
    print(f'\n  Tipos de dato:')
    for dtype, count in df.dtypes.value_counts().items():
        print(f'    {str(dtype):<12}: {count} columna(s)')
    print()
    display(df.head(3))


def grafica_nulos(df, titulo, color='#003DA5'):
    """Gráfica horizontal de % de nulos por columna."""
    nulos_pct = (df.isnull().sum() / len(df) * 100).sort_values(ascending=True)
    nulos_pct = nulos_pct[nulos_pct > 0]  # Solo columnas con nulos
    if nulos_pct.empty:
        print(f'  [{titulo}] Sin valores nulos.')
        return
    colores_bar = ['#C8102E' if v > 50 else color for v in nulos_pct.values]
    fig, ax = plt.subplots(figsize=(10, max(3, len(nulos_pct)*0.4)))
    bars = ax.barh(nulos_pct.index, nulos_pct.values, color=colores_bar)
    ax.xaxis.set_major_formatter(mtick.PercentFormatter())
    ax.set_xlim(0, 105)
    for bar, val in zip(bars, nulos_pct.values):
        ax.text(val + 0.5, bar.get_y() + bar.get_height()/2,
                f'{val:.1f}%', va='center', fontsize=9)
    ax.set_title(f'Porcentaje de Valores Nulos — {titulo}', fontweight='bold')
    ax.set_xlabel('% de nulos')
    plt.tight_layout()
    plt.show()


def grafica_tipos(df, titulo):
    """Pie chart de tipos de dato."""
    conteo = df.dtypes.astype(str).value_counts()
    fig, ax = plt.subplots(figsize=(5, 4))
    ax.pie(conteo.values, labels=conteo.index, autopct='%1.0f%%',
           colors=COLORES[:len(conteo)], startangle=90)
    ax.set_title(f'Tipos de dato — {titulo}', fontweight='bold')
    plt.tight_layout()
    plt.show()

---
## 3. Perfilado — Fuente 1: LigaMX (JSON)

**Dataset:** Partidos de la Liga MX 2016–2024  
**Problemática relacionada:** #4 — Inconsistencia en nombres de equipos entre temporadas  

In [ ]:
perfil_basico(df_ligamx, 'LigaMX 2016-2024')

In [ ]:
grafica_nulos(df_ligamx, 'LigaMX')
grafica_tipos(df_ligamx, 'LigaMX')

In [ ]:
# Distribución de partidos por temporada
col_temporada = 'season' if 'season' in df_ligamx.columns else df_ligamx.columns[0]
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

conteo_temp = df_ligamx[col_temporada].value_counts().sort_index()
axes[0].bar(conteo_temp.index.astype(str), conteo_temp.values, color='#003DA5')
axes[0].set_title('Partidos por Temporada — LigaMX', fontweight='bold')
axes[0].set_xlabel('Temporada')
axes[0].set_ylabel('Número de partidos')
axes[0].tick_params(axis='x', rotation=45)

# Equipos con más apariciones (detección de inconsistencia en nombres)
col_home = [c for c in df_ligamx.columns if 'home' in c.lower() or 'local' in c.lower()][0]
top_equipos = df_ligamx[col_home].value_counts().head(15)
axes[1].barh(top_equipos.index[::-1], top_equipos.values[::-1], color='#C8102E')
axes[1].set_title('Top 15 Equipos (local) — LigaMX', fontweight='bold')
axes[1].set_xlabel('Apariciones')

plt.tight_layout()
plt.show()

# Alerta: equipos únicos (posibles inconsistencias de nombres)
n_equipos = df_ligamx[col_home].nunique()
print(f'\nAlerta de calidad — Consistencia (Problemática #4):')
print(f'  Equipos únicos detectados en columna "{col_home}": {n_equipos}')
print(f'  Liga MX tiene 18 equipos activos — valores mayores sugieren inconsistencias de nombres.')
print(f'\n  Muestra de nombres únicos:')
print(df_ligamx[col_home].unique()[:20])

---
## 4. Perfilado — Fuente 2: Transfermarkt players (XLSX)

**Dataset:** Jugadores con valor de mercado y estadísticas reales  
**Problemáticas relacionadas:** #2 (valores faltantes/desactualizados), #3 (jugadores duplicados)

In [ ]:
perfil_basico(df_tm, 'Transfermarkt — Players')

In [ ]:
grafica_nulos(df_tm, 'Transfermarkt Players')
grafica_tipos(df_tm, 'Transfermarkt Players')

In [ ]:
# Análisis de valor de mercado
col_valor = [c for c in df_tm.columns if 'market' in c.lower() or 'value' in c.lower() or 'market_value' in c.lower()]
col_pos   = [c for c in df_tm.columns if 'posit' in c.lower() or 'sub_position' in c.lower()]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

if col_valor:
    col_v = col_valor[0]
    valores = df_tm[col_v].dropna()
    axes[0].hist(np.log1p(valores), bins=50, color='#003DA5', edgecolor='white')
    axes[0].set_title(f'Distribución de {col_v} (log)', fontweight='bold')
    axes[0].set_xlabel('log(1 + valor de mercado)')
    axes[0].set_ylabel('Frecuencia')
    pct_nulos_valor = df_tm[col_v].isnull().mean() * 100
    axes[0].set_xlabel(f'log(valor)  |  Nulos: {pct_nulos_valor:.1f}%')
else:
    axes[0].text(0.5, 0.5, 'Columna de valor no encontrada', ha='center', transform=axes[0].transAxes)

if col_pos:
    col_p = col_pos[0]
    top_pos = df_tm[col_p].value_counts().head(12)
    axes[1].barh(top_pos.index[::-1], top_pos.values[::-1], color='#C8102E')
    axes[1].set_title(f'Distribución por {col_p}', fontweight='bold')
    axes[1].set_xlabel('Jugadores')
else:
    axes[1].text(0.5, 0.5, 'Columna de posición no encontrada', ha='center', transform=axes[1].transAxes)

plt.tight_layout()
plt.show()

# Alerta de calidad: jugadores con múltiples nacionalidades (Problemática #3)
col_nac = [c for c in df_tm.columns if 'nation' in c.lower() or 'country' in c.lower()]
if col_nac:
    print(f'\nAlerta de calidad — Unicidad (Problemática #3):')
    print(f'  Columnas de nacionalidad detectadas: {col_nac}')
    for c in col_nac[:2]:
        print(f'  Valores únicos en "{c}": {df_tm[c].nunique()}')

---
## 5. Perfilado — Fuente 3: FIFA 23 male_coaches (TXT)

**Dataset:** Entrenadores masculinos FIFA 23  
**Problemática relacionada:** #1 (inconsistencia en nombres), #5 (métricas incomparables)

In [ ]:
perfil_basico(df_fifa, 'FIFA 23 — Male Coaches')

In [ ]:
grafica_nulos(df_fifa, 'FIFA 23 Male Coaches')
grafica_tipos(df_fifa, 'FIFA 23 Male Coaches')

In [ ]:
# Análisis de rangos numéricos (escala 0-100 FIFA — Problemática #5)
cols_num = df_fifa.select_dtypes(include=[np.number]).columns.tolist()
print(f'Columnas numéricas en FIFA23 male_coaches: {cols_num}')

if cols_num:
    fig, axes = plt.subplots(1, min(len(cols_num), 3), figsize=(14, 4))
    if len(cols_num) == 1:
        axes = [axes]
    for ax, col in zip(axes, cols_num[:3]):
        df_fifa[col].dropna().hist(bins=30, ax=ax, color='#003DA5', edgecolor='white')
        ax.set_title(col, fontweight='bold')
        ax.set_ylabel('Frecuencia')
    plt.suptitle('Distribución de variables numéricas — FIFA 23 (escala 0–100)', fontweight='bold')
    plt.tight_layout()
    plt.show()

# Estadísticas descriptivas
display(df_fifa.describe())

---
## 6. Perfilado — Fuente 4: OpenPublicDomainFootballData (CSV)

**Dataset:** Partidos históricos Liga MX desde múltiples temporadas  
**Problemática relacionada:** #4 (cambios de nombre de equipos: Monarcas → Mazatlán)

In [ ]:
perfil_basico(df_opfd, 'OpenPublicDomain — Liga MX')

In [ ]:
grafica_nulos(df_opfd, 'OpenPublicDomain LigaMX')
grafica_tipos(df_opfd, 'OpenPublicDomain LigaMX')

In [ ]:
# Análisis de resultados y equipos
col_t1 = [c for c in df_opfd.columns if 'team' in c.lower() or 'home' in c.lower() or 'Team 1' in c][0]
col_ft = [c for c in df_opfd.columns if 'ft' in c.lower() or 'score' in c.lower() or 'FT' in c][0]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Equipos únicos (cambios de nombre a lo largo de temporadas)
top_equipos_opfd = df_opfd[col_t1].value_counts().head(20)
axes[0].barh(top_equipos_opfd.index[::-1], top_equipos_opfd.values[::-1], color='#003DA5')
axes[0].set_title('Frecuencia de Equipos (local) — Histórico', fontweight='bold')
axes[0].set_xlabel('Partidos')

# Distribución de resultados
resultados = df_opfd[col_ft].value_counts().head(15)
axes[1].bar(resultados.index.astype(str), resultados.values, color='#C8102E')
axes[1].set_title('Top 15 Resultados más frecuentes', fontweight='bold')
axes[1].set_xlabel('Resultado (FT)')
axes[1].set_ylabel('Partidos')
axes[1].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.show()

# Alerta: número de equipos únicos vs. temporadas (detectar cambios de nombre)
n_eq = df_opfd[col_t1].nunique()
print(f'\nAlerta de calidad — Consistencia (Problemática #4):')
print(f'  Equipos únicos en histórico: {n_eq}')
print(f'  Equipos esperados (~18 activos + históricos): ~25-30')
if n_eq > 30:
    print(f'  ALERTA: {n_eq} nombres únicos sugieren inconsistencias (ej. Monarcas Morelia vs Mazatlán FC)')

---
## 7. Resumen Comparativo de Calidad

Tabla consolidada de métricas de calidad por fuente, siguiendo el framework DAMA-DMBOK.

In [ ]:
def metricas_calidad(df, nombre, formato):
    total_celdas = df.shape[0] * df.shape[1]
    return {
        'Fuente'          : nombre,
        'Formato'         : formato,
        'Filas'           : df.shape[0],
        'Columnas'        : df.shape[1],
        'Nulos (%)'       : round(df.isnull().sum().sum() / total_celdas * 100, 2),
        'Duplicados (%)'  : round(df.duplicated().sum() / df.shape[0] * 100, 2),
        'Cols numéricas'  : df.select_dtypes(include=np.number).shape[1],
        'Cols texto'      : df.select_dtypes(include='object').shape[1],
    }

resumen = pd.DataFrame([
    metricas_calidad(df_ligamx, 'LigaMX 2016-2024',  'JSON'),
    metricas_calidad(df_tm,     'Transfermarkt',     'XLSX'),
    metricas_calidad(df_fifa,   'FIFA23 coaches',    'TXT'),
    metricas_calidad(df_opfd,   'OpenPublicDomain',  'CSV'),
])

# Formato visual con semáforo en columnas clave
display(resumen.style
    .background_gradient(subset=['Nulos (%)'],     cmap='RdYlGn_r', vmin=0, vmax=100)
    .background_gradient(subset=['Duplicados (%)'], cmap='RdYlGn_r', vmin=0, vmax=20)
    .set_caption('Resumen de Calidad de Datos por Fuente — Framework DAMA-DMBOK')
    .format({'Filas': '{:,}', 'Nulos (%)': '{:.1f}%', 'Duplicados (%)': '{:.1f}%'})
)

In [ ]:
# Gráfica comparativa de completitud entre fuentes
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# % Nulos por fuente
axes[0].bar(resumen['Fuente'], resumen['Nulos (%)'],
            color=['#003DA5', '#C8102E', '#888888', '#004F9E'])
axes[0].set_title('% Nulos por Fuente', fontweight='bold')
axes[0].set_ylabel('% de valores nulos')
axes[0].tick_params(axis='x', rotation=15)
for i, v in enumerate(resumen['Nulos (%)']):
    axes[0].text(i, v + 0.3, f'{v:.1f}%', ha='center', fontweight='bold')

# % Duplicados por fuente
axes[1].bar(resumen['Fuente'], resumen['Duplicados (%)'],
            color=['#003DA5', '#C8102E', '#888888', '#004F9E'])
axes[1].set_title('% Duplicados por Fuente', fontweight='bold')
axes[1].set_ylabel('% de filas duplicadas')
axes[1].tick_params(axis='x', rotation=15)
for i, v in enumerate(resumen['Duplicados (%)']):
    axes[1].text(i, v + 0.05, f'{v:.1f}%', ha='center', fontweight='bold')

plt.suptitle('Comparativa de Calidad entre Fuentes — Cruz Azul Intelligence', fontweight='bold', fontsize=14)
plt.tight_layout()
plt.savefig(os.path.join(REPORTS, 'comparativa_calidad_fuentes.png'), dpi=150, bbox_inches='tight')
plt.show()
print('Gráfica guardada en reports/')

---
## 8. Perfilado Automático con ydata-profiling

Se genera un reporte HTML interactivo por cada fuente con visualizaciones automáticas de distribuciones, correlaciones y alertas de calidad.

In [ ]:
from ydata_profiling import ProfileReport

fuentes = [
    (df_ligamx, 'LigaMX_2016-2024',   'LigaMX — Partidos 2016-2024'),
    (df_tm,     'Transfermarkt_players', 'Transfermarkt — Jugadores'),
    (df_fifa,   'FIFA23_male_coaches',   'FIFA 23 — Entrenadores Masculinos'),
    (df_opfd,   'OpenPublicDomain',      'OpenPublicDomain — Liga MX Histórico'),
]

for df, nombre_archivo, titulo in fuentes:
    print(f'Generando reporte: {titulo} ...')
    reporte = ProfileReport(
        df,
        title=f'Perfilado — {titulo}',
        explorative=True,
        minimal=False
    )
    ruta_html = os.path.join(REPORTS, f'perfil_{nombre_archivo}.html')
    reporte.to_file(ruta_html)
    print(f'  Guardado: {ruta_html}')

print('\nTodos los reportes generados en la carpeta reports/')

---
## 9. Conclusiones del Perfilado

Resumen de hallazgos por dimensión de calidad DAMA-DMBOK para cada fuente:

In [ ]:
conclusiones = {
    'Fuente': ['LigaMX', 'Transfermarkt', 'FIFA 23', 'OpenPublicDomain'],
    'Completitud': [
        'Evaluar % nulos en columnas de goles y árbitros',
        'Valores de mercado faltantes en jugadores de Liga MX (Prob. #2)',
        'Verificar columnas de rating y nacionalidad',
        'Tiempo y zona horaria frecuentemente ausentes'
    ],
    'Consistencia': [
        'Nombres de equipos cambian entre temporadas (Prob. #4)',
        'Nombres de jugadores con múltiples formatos (Prob. #1)',
        'Escalas FIFA 0-100 incomparables con Transfermarkt (Prob. #5)',
        'Nombres de equipos históricos vs actuales (Prob. #4)'
    ],
    'Unicidad': [
        'Revisar partidos duplicados en períodos superpuestos',
        'Jugadores naturalizados con doble registro (Prob. #3)',
        'Entrenadores con variaciones de nombre',
        'Partidos repetidos al concatenar temporadas'
    ],
    'Acción siguiente': [
        'Estandarizar nombres de equipos',
        'Imputar valores de mercado faltantes',
        'Normalizar escalas a rango común',
        'Deduplicar por fecha + equipos'
    ]
}

df_conclusiones = pd.DataFrame(conclusiones)
display(df_conclusiones.style
    .set_caption('Conclusiones del Perfilado — Acciones para etapa de Limpieza')
    .set_properties(**{'text-align': 'left', 'white-space': 'pre-wrap'})
)

print('\nSiguiente etapa: limpieza.ipynb')